# 1. YouTube Video Frame Capture - Setup, Code, & Docs

⚠️ **Warning: This dialog requires `node` (Node.js), `ffmpeg`, and `yt-dlp` to work properly.**

- **Node.js**: Required by `yt-dlp` to resolve YouTube's obfuscated stream URLs (720p DASH).
- **ffmpeg**: Required to extract video frames from streams. Install with `brew install ffmpeg`.
- **yt-dlp**: YouTube metadata and stream URL extraction (bundled with dialeng).

See the docs in section 1.3 for more details.

## 1.1. Usage

**First-time setup (once per instance):** Uncollapse section 1.2, uncomment and run the code cells there to install `srt` and `deno`. Run the two test cells to confirm everything works.

**Getting started:**
1. Uncollapse section 1 and click the **Run All** toolbar icon to execute all code cells.
2. In **section 2**, edit `yt_video_id` with your YouTube video ID, and optionally set `yt_playlist_id` (leave it as `''` if you don't need a playlist). Run the two cells in section 2.
3. Navigate to any point in the video, or select another video from the playlist using the player controls.
4. Use the **⊙ capture button** in the toolbar to pause the video and capture the current frame. Use the **▾ dropdown** to choose the capture mode (defaults to **Send to Prompt & Run**) and to customise the prompt text sent with each capture.

**Tips:**
- Set a **bookmark** (1–9) on the YouTube player cell in section 2, so you can jump back to the video quickly with a keyboard shortcut. Note: A bookmark might already exist set via the template.
- After capturing, press **End** to jump to the bottom of the dialog.
- If captures start failing after a while (expired stream URLs), run `_stream_cache.clear()` in a code cell to force a fresh resolve.

##  1.2. Dependencies

Uncomment the code message below to install `srt` if not already done so.

In [ ]:
# !pip install srt

Uncomment the code messages below (section 1.2 only), and run, to verify `node` and `yt-dlp` are working correctly.

In [ ]:
# Verify dependencies are available
import subprocess
print('node:', subprocess.check_output(['node', '--version'], text=True).strip())
print('ffmpeg:', subprocess.check_output(['ffmpeg', '-version'], text=True).split('\n')[0])
print('yt-dlp:', subprocess.check_output(['yt-dlp', '--version'], text=True).strip())

In [ ]:
# Add node (nvm) and ffmpeg (homebrew) to kernel PATH so yt-dlp can find them
import os, shutil
nvm_node = os.path.expanduser('~/.nvm/versions/node')
if os.path.isdir(nvm_node):
    versions = sorted(os.listdir(nvm_node))
    if versions:
        node_bin = os.path.join(nvm_node, versions[-1], 'bin')
        if node_bin not in os.environ['PATH']:
            os.environ['PATH'] = node_bin + ':' + os.environ['PATH']
            print(f'Added {node_bin} to PATH')
# Also ensure Homebrew paths are available
for p in ['/opt/homebrew/bin', '/usr/local/bin']:
    if os.path.isdir(p) and p not in os.environ['PATH']:
        os.environ['PATH'] = p + ':' + os.environ['PATH']
        print(f'Added {p} to PATH')
print('node:', shutil.which('node') or 'NOT FOUND')
print('ffmpeg:', shutil.which('ffmpeg') or 'NOT FOUND')
print('yt-dlp:', shutil.which('yt-dlp') or 'NOT FOUND')

In [ ]:
# Test: Can yt-dlp resolve a 720p DASH URL using Node.js?
import subprocess
print(subprocess.check_output(['yt-dlp', '-g', '-f', '136', 'https://youtu.be/8iGzBMboA0I'], text=True, timeout=15)[:80])

In [ ]:
# Test: Can yt-dlp resolve a 720p DASH URL?
import subprocess
print(subprocess.check_output(['yt-dlp', '-g', '-f', '136', 'https://youtu.be/8iGzBMboA0I'], text=True, timeout=15)[:80])

## 1.3. Docs

**What this does**: Embeds a YouTube player in the dialog with a toolbar ⊙ button. When clicked, it pauses the video, captures the current frame via `yt-dlp` + `ffmpeg`, gets the transcript around that timestamp, creates a prompt (or note) message with the frame image and transcript, optionally auto-runs the prompt, then deletes the trigger code cell.

**Architecture**: The YouTube player is embedded via the IFrame API (JS). The ⊙ toolbar button (JS) creates and runs a code cell that calls `yt_capture` (async Python), which resolves a stream URL via `yt-dlp`, extracts a frame with `ffmpeg`, fetches subtitles via `yt-dlp` + `srt` parsing, and manages messages via `_add_msg_unsafe` and solveit's internal endpoints.

**JS runtime dependency**: A JavaScript runtime (Node.js or Deno) is required by `yt-dlp` to resolve YouTube's obfuscated DASH stream URLs. With a JS runtime available, frame capture uses **720p** DASH format (format 136) and fast **random-access** byte-range seeking via MP4 `sidx` segment index (~0.4s per frame, ~500KB download regardless of position). Without one, capture falls back to **360p** progressive format (format 18) with slow linear `ffmpeg -ss` seeking, which can hang or timeout for timestamps deep into a video. Section 1.2 has a test cell to verify your setup.

**Transcript capture**: `get_video_context` grabs subtitles within **±15 seconds** of the captured frame's timestamp (controlled by `delta=15` in `yt_capture`). Subtitles are downloaded once via `yt-dlp` (SRT format) and cached to disk; subsequent captures reuse the cached file.

**Capture modes**: The ⊙ toolbar button has a dropdown (▾) with three modes: **Send to Prompt & Run** (default, this creates a prompt with the frame and transcript, and auto-runs it), **Send to Prompt** (creates the prompt but doesn't run it), and **Send to Note** (creates a plain note with the frame and transcript, no AI). The dropdown also has a text field to customise the prompt text sent with each capture.

**Navigation tip**: For the best workflow, set a **bookmark** (1–9) on the YouTube player code cell (section 2). Then use the bookmark keyboard shortcut to jump straight back to the video whenever you want to scrub to a new frame and capture it. After hitting the ⊙ capture button, press **End** to jump to the bottom of the dialog manually if needed, though the dialog should auto-scroll once the captured frame and prompt have been inserted.

**Switching videos**: Change `yt_video_id` (and optionally `yt_playlist_id`) in section 2 and re-run the cell. The old player is automatically destroyed and replaced, no need to duplicate the cell.

**Playlist support**: You can optionally set `yt_playlist_id` in section 2 to load a YouTube playlist. Once loaded, you can switch between any video in the playlist using the player's built-in controls and capture frames seamlessly, no reload or re-run needed. If you don't want a playlist or one isn't available for the video, just set `yt_playlist_id = ''`.

**Frame accuracy**: Frame extraction assumes **30fps** video. The exact frame is selected by calculating `(timestamp - segment_start) * 30`. If a video uses a different frame rate, the captured frame may be slightly offset from the exact timestamp.

**Troubleshooting**: Stream URLs are cached per video ID in `_stream_cache`. If captures start failing after a while (expired URLs), run `_stream_cache.clear()` in a code cell to force a fresh resolve.

## 1.4 Core Code

### 1.4.1 Subtitle / Transcript Parsing

`parse_srt` reads an SRT subtitle file (preferring Chinese-English bilingual if available). `get_video_context` uses `yt-dlp` to download subtitles for a YouTube video, then extracts and deduplicates the transcript text within a time window around a given timestamp.

In [ ]:
import srt, yt_dlp, subprocess, httpx, os
from pathlib import Path
from typing import Annotated
from bisect import bisect_left, bisect_right

def parse_srt(srt_path: Path):
    """Parse an SRT subtitle file, preferring zh-en bilingual if available."""
    zh_en = srt_path.with_name(srt_path.name.replace('.en.srt', '.zh-en.srt'))
    lang, path = ('zh', zh_en) if zh_en.exists() else ('en', srt_path)
    if not path.exists(): return 'en', []
    return lang, list(srt.parse(path.read_text()))

def get_video_context(
    url: Annotated[str, "URL of the YouTube video"],
    currtime: Annotated[float, "Current playback time in the video, in seconds"],
    delta: Annotated[int, "Seconds before/after currtime, defaults to 60 seconds"]=60
):
    """Get subtitle context around a specific timestamp in a YouTube video"""
    currtime = float(currtime)
    delta = int(delta)

    opts = {
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['en'],
        'subtitlesformat': 'srt',
        'skip_download': True,
        'nooverwrites': True,
        'outtmpl': 'yt_data/subtitles/%(title)s',
        'quiet': True,
        'no_warnings': True
    }

    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        sub_path = Path(f"{ydl.prepare_filename(info)}.en.srt")

    lang, subtitles = parse_srt(sub_path)
    # Slice subtitles within [currtime-delta, currtime+delta], back one entry for overlap
    lo = max(0, bisect_left(subtitles, max(0, currtime-delta), key=lambda s: s.start.total_seconds()) - 1)
    hi = bisect_right(subtitles, min(info['duration'], currtime+delta), key=lambda s: s.start.total_seconds())
    texts = [s.content.split('\n')[0] if lang == 'zh' else s.content for s in subtitles[lo:hi]]
    return ' '.join(dict.fromkeys(t.strip() for t in texts if t.strip()))  # deduplicated

### 1.4.2 Frame Capture

`yt_capture` is an async function that orchestrates the full capture pipeline: resolves a stream URL via `yt-dlp` (with caching and 720p→360p fallback), extracts a single frame with `ffmpeg`, fetches the surrounding transcript, creates a prompt or note message with the image and transcript attached, then deletes its own trigger code cell.

In [ ]:
from dialoghelper.core import _add_msg_unsafe, call_endpa, dh_settings
dh_settings['port'] = 8000  # dialeng runs on port 8000 (dialoghelper defaults to 5001)
import struct
from bisect import bisect_left
from urllib.parse import urlparse

# Cache stream URLs and segment index by video ID to avoid repeated network calls
_stream_cache = {}

_FMT_LABELS = {'136': '720p', '18': '360p'}
_MODE_ARGS  = {
    'capture':    dict(msg_type='note',   placement='at_end'),
    'prompt':     dict(msg_type='prompt', placement='at_end'),
    'prompt_run': dict(msg_type='prompt', placement='at_end', run_mode='run'),
}

async def yt_capture(url, timestamp, code_cell_id, prompt_text="Describe this video frame.", mode="prompt_run"):
    """Capture frame, get transcript, create prompt/note with image, delete this cell.
    mode: 'prompt_run' (create prompt & run), 'prompt' (create prompt only), 'capture' (note, no AI)"""
    log = lambda msg: print(f'[yt_capture] {msg}')
    timestamp = float(timestamp)

    # 1a. Resolve stream URL + segment index (cached per video ID)
    vid = urlparse(url).path.lstrip('/')
    if vid not in _stream_cache:
        log(f'Resolving stream URL for {vid}...')
        stream_url = None
        for fmt in ('136', '18'):
            try:
                stream_url = subprocess.check_output(
                    ['yt-dlp', '-g', '-f', fmt, url], text=True, timeout=15).strip()
                log(f'Using format {fmt} ({_FMT_LABELS[fmt]})')
                break
            except (subprocess.TimeoutExpired, subprocess.CalledProcessError):
                log(f'Format {fmt} failed, trying next...')
        if not stream_url:
            log('ERROR: No stream URL could be resolved'); return

        # Parse MP4 box structure to find sidx (segment index)
        log('Parsing segment index...')
        head = httpx.get(f"{stream_url}&range=0-50000", timeout=10).content
        pos = 0
        sidx_pos = sidx_size = 0
        while pos < len(head) - 8:
            size     = int.from_bytes(head[pos:pos+4], 'big')
            box_type = head[pos+4:pos+8].decode('ascii', errors='replace')
            if box_type == 'sidx':
                sidx_pos, sidx_size = pos, size; break
            if size < 8: break
            pos += size

        if not sidx_size:
            log('No sidx box found, falling back to direct seek')
            _stream_cache[vid] = {'stream': stream_url, 'segments': None, 'init_end': 0}
        else:
            sidx_data  = head[sidx_pos:sidx_pos+sidx_size]
            init_end   = sidx_pos - 1
            data_start = sidx_pos + sidx_size

            # Parse sidx entries (version 0: all fixed-width fields)
            off = 16   # box header (8) + version+flags (4) + reference_ID (4)
            timescale, = struct.unpack_from('>I', sidx_data, off); off += 4
            off += 8   # earliest_presentation_time (4) + first_offset (4)
            off += 2   # reserved
            seg_count, = struct.unpack_from('>H', sidx_data, off); off += 2

            segments = []
            t_accum, byte_accum = 0.0, data_start
            for _ in range(seg_count):
                ref, = struct.unpack_from('>I', sidx_data, off); off += 4
                dur, = struct.unpack_from('>I', sidx_data, off); off += 4
                off += 4   # SAP flags
                ref_size = ref & 0x7FFFFFFF
                seg_end  = t_accum + dur / timescale
                segments.append((t_accum, seg_end, byte_accum, ref_size))
                t_accum    = seg_end
                byte_accum += ref_size

            _stream_cache[vid] = {'stream': stream_url, 'segments': segments, 'init_end': init_end}
            log(f'Indexed {seg_count} segments')

    cache  = _stream_cache[vid]
    stream = cache['stream']
    log('Stream URL ready')

    # 1b. Extract frame via byte-range + sidx (fast) or direct seek (fallback)
    os.makedirs('yt_frames', exist_ok=True)
    fname      = f'frame_{timestamp}.jpg'
    frame_path = f'yt_frames/{fname}'
    log(f'Extracting frame at t={timestamp}...')

    try:
        if cache['segments']:
            # Find segment containing timestamp via bisect on segment end-times
            ends = [s[1] for s in cache['segments']]
            idx  = bisect_left(ends, timestamp)
            if idx >= len(ends):
                log(f'ERROR: timestamp {timestamp} out of range'); return
            seg_start, _, byte_start, byte_size = cache['segments'][idx]

            frame_n = int((timestamp - seg_start) * 30)
            init  = httpx.get(f"{stream}&range=0-{cache['init_end']}", timeout=10).content
            chunk = httpx.get(f"{stream}&range={byte_start}-{byte_start+byte_size}", timeout=10).content
            Path('/tmp/partial.mp4').write_bytes(init + chunk)
            subprocess.run(['ffmpeg', '-i', '/tmp/partial.mp4',
                '-vf', f"select='eq(n,{frame_n})'", '-frames:v', '1', '-vsync', 'vfr', '-q:v', '2',
                frame_path, '-y'], capture_output=True, timeout=10)
        else:
            # Fallback: direct seek (slow for large timestamps)
            subprocess.run(['ffmpeg', '-ss', str(timestamp), '-i', stream, '-frames:v', '1', '-q:v', '2',
                frame_path, '-y'], capture_output=True, timeout=30)
    except (subprocess.TimeoutExpired, httpx.TimeoutException) as e:
        log(f'ERROR: frame extraction failed: {e}'); return

    # 2. Get transcript
    log('Fetching transcript...')
    transcript = get_video_context(url=url, currtime=timestamp, delta=15)

    # 3. Build content
    hrs, rem   = divmod(int(timestamp), 3600)
    mins, secs = divmod(rem, 60)
    parts = [f'![{fname}]({frame_path})']
    if prompt_text: parts.append(prompt_text)
    parts.append(f'Transcript (~{hrs}:{mins:02d}:{secs:02d}): **{transcript}**')
    content = '\n\n'.join(parts)

    # 4. Create message and delete trigger cell
    log(f'Creating {mode} message...')
    await _add_msg_unsafe(content=content, **_MODE_ARGS[mode])
    log('Cleaning up trigger cell...')
    await call_endpa('rm_msg_', msid=code_cell_id)

### 1.4.3 Toolbar Capture Button

Injects a split ⊙ button into the solveit nav bar using the YouTube IFrame API. The main button pauses the video and triggers a frame capture. The dropdown lets you choose the mode (prompt & run, prompt only, or note) and customise the prompt text.

In [ ]:
from IPython.display import HTML
HTML("""<script>
(function() {
    const nav = document.querySelector('.toolbar-right');
    if (!nav) return console.warn('Nav toolbar not found');

    // Remove old elements and click listener
    document.getElementById('yt-capture-wrap')?.remove();
    if (window._ytCapClickOutside) document.removeEventListener('click', window._ytCapClickOutside);

    // Inject hover styles once
    if (!document.getElementById('yt-capture-style')) {
        const style = document.createElement('style');
        style.id = 'yt-capture-style';
        style.textContent = '.yt-mode-item:hover { background: #f0f4ff !important; }';
        document.head.appendChild(style);
    }

    // Persistent settings
    if (!window.ytCaptureSettings) window.ytCaptureSettings = {
        promptText: 'Describe this video frame.',
        mode: 'prompt_run'  // 'prompt_run', 'prompt', 'capture'
    };
    const S = window.ytCaptureSettings;

    // Icon helper: circle with mode-dependent color/fill
    const captureIcon = (color, filled) => {
        const fill = filled ? color : 'none';
        return `<svg viewBox="0 0 16 16" width="16" height="16" style="vertical-align:middle"><circle cx="8" cy="8" r="5.5" stroke="${color}" stroke-width="1.5" fill="${fill}"/></svg>`;
    };
    const chevron = `<svg viewBox="0 0 24 24" width="12" height="12" fill="currentColor"><path fill-rule="evenodd" clip-rule="evenodd" d="M4.29289 8.29289C4.68342 7.90237 5.31658 7.90237 5.70711 8.29289L12 14.5858L18.2929 8.29289C18.6834 7.90237 19.3166 7.90237 19.7071 8.29289C20.0976 8.68342 20.0976 9.31658 19.7071 9.70711L12.7071 16.7071C12.3166 17.0976 11.6834 17.0976 11.2929 16.7071L4.29289 9.70711C3.90237 9.31658 3.90237 8.68342 4.29289 8.29289Z"/></svg>`;

    const modeStyles = {
        prompt_run: { color: '#ff5555', filled: true,  label: 'Send to Prompt &amp; Run' },
        prompt:     { color: '#ff5555', filled: false, label: 'Send to Prompt' },
        capture:    { color: '#22dd66', filled: true,  label: 'Send to Note' }
    };

    // Container
    const wrap = document.createElement('div');
    wrap.id = 'yt-capture-wrap';
    wrap.style.cssText = 'display:flex;position:relative;';

    // Main capture button
    const btn = document.createElement('button');
    btn.className = 'btn btn-sm';
    btn.style.cssText = 'border-radius:4px 0 0 4px;border-right:none;';
    btn.title = 'Capture video frame';

    // Drop arrow button
    const dd = document.createElement('button');
    dd.className = 'btn btn-sm';
    dd.style.cssText = 'border-radius:0 4px 4px 0;padding:0 4px;margin-left:0;';
    dd.innerHTML = chevron;
    dd.title = 'Capture settings';

    function updateMainIcon() {
        const m = modeStyles[S.mode];
        btn.innerHTML = captureIcon(m.color, m.filled);
    }
    updateMainIcon();

    // Dropdown panel
    const panel = document.createElement('div');
    panel.style.cssText = 'display:none;position:absolute;top:100%;right:0;background:white;border:1px solid #ccc;border-radius:4px;box-shadow:0 2px 8px rgba(0,0,0,0.15);z-index:10;min-width:220px;margin-top:2px;overflow:hidden;';

    function buildPanel() {
        let html = `<div style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">
            <textarea id="yt-prompt-text" rows="2" placeholder="Prompt text..."
                style="width:100%;background:#f9fafb;color:#333;border:1px solid #d1d5db;border-radius:4px;padding:6px 8px;font-size:12px;resize:vertical;font-family:inherit;box-sizing:border-box;">${S.promptText}</textarea>
        </div>`;
        for (const [mode, ms] of Object.entries(modeStyles)) {
            const active = mode === S.mode;
            const bg     = active ? '#f0f4ff' : '#fff';
            const weight = active ? '600' : '400';
            html += `<div class="yt-mode-item" data-mode="${mode}"
                style="padding:6px 12px;cursor:pointer;display:flex;align-items:center;gap:8px;background:${bg};font-weight:${weight};font-size:13px;color:#333;white-space:nowrap;">
                ${captureIcon(ms.color, ms.filled)}
                <span>${ms.label}</span>
            </div>`;
        }
        panel.innerHTML = html;
        panel.querySelector('#yt-prompt-text')?.addEventListener('input', e => { S.promptText = e.target.value; });
    }
    buildPanel();

    dd.onclick = (e) => {
        e.stopPropagation();
        const show = panel.style.display === 'none';
        panel.style.display = show ? 'block' : 'none';
        if (show) buildPanel();
    };

    panel.addEventListener('click', e => {
        e.stopPropagation();
        const item = e.target.closest('.yt-mode-item');
        if (item) {
            S.mode = item.dataset.mode;
            updateMainIcon();
            panel.style.display = 'none';
        }
    });

    window._ytCapClickOutside = () => { panel.style.display = 'none'; };
    document.addEventListener('click', window._ytCapClickOutside);

    // FormData POST helper
    const post = (path, fields) => {
        const fd = new FormData();
        for (const [k, v] of Object.entries(fields)) fd.append(k, v);
        return fetch(path, {method: 'POST', body: fd});
    };

    // Capture action
    async function doCapture() {
        if (!window.ytPlayer?.getCurrentTime) return alert('No video loaded');
        ytPlayer.pauseVideo();
        const t   = ytPlayer.getCurrentTime().toFixed(2);
        const dlg = window.NOTEBOOK_ID;
        const url = `https://youtu.be/${ytPlayer.getVideoData().video_id}`;
        const promptEsc = JSON.stringify(S.promptText).slice(1, -1).replace(/`/g, '\\`');
        btn.style.opacity = '0.4';
        console.log('[YT-CAP] dlg=' + dlg + ' t=' + t + ' mode=' + S.mode);

        try {
            const resp1 = await post('/add_relative_', {dlg_name: dlg, content: 'pass', msg_type: 'code', placement: 'at_end'});
            const text1 = await resp1.text();
            console.log('[YT-CAP] add_relative_ status=' + resp1.status + ' body=' + text1);
            let res;
            try { res = JSON.parse(text1); } catch(e) {
                console.error('[YT-CAP] add_relative_ response is not JSON:', text1.substring(0, 200));
                alert('add_relative_ returned non-JSON: ' + text1.substring(0, 100));
                return;
            }
            console.log('[YT-CAP] cell id=' + res.id);
            const code = `await yt_capture("${url}", ${t}, "${res.id}", "${promptEsc}", "${S.mode}")`;
            console.log('[YT-CAP] code=' + code);

            const resp2 = await post('/update_msg_', {dlg_name: dlg, id_: res.id, content: code});
            const text2 = await resp2.text();
            console.log('[YT-CAP] update_msg_ status=' + resp2.status + ' body=' + text2.substring(0, 200));

            const resp3 = await post('/add_runq_', {dlg_name: dlg, ids: res.id});
            const text3 = await resp3.text();
            console.log('[YT-CAP] add_runq_ status=' + resp3.status + ' body=' + text3.substring(0, 200));
        } catch(e) {
            console.error('[YT-CAP] Capture failed:', e);
            alert('Capture failed: ' + e.message);
        } finally {
            btn.style.opacity = '';
        }
    }

    btn.onclick = doCapture;

    wrap.appendChild(btn);
    wrap.appendChild(dd);
    wrap.appendChild(panel);
    nav.prepend(wrap);
    console.log('YT capture split button added');
})();
</script>""")

HTML(<script>
(function() {
    const nav = document.querySelector('nav .flex.flex-wrap.justify-end');
    if (!nav) return console.warn('Nav toolbar not found');

    // Remove old elements and click listener
    document.getElementById('yt-capture-wrap')?.remove();
    if (window._ytCapClickOutside) document.removeEventListener('click', window._ytCapClickOutside);

    // Inject hover styles once
    if (!document.getElementById('yt-capture-style')) {
        const style = document.createElement('style');
        style.id = 'yt-capture-style';
        style.textContent = '.yt-mode-item:hover { background: #f0f4ff !important; }';
        document.head.appendChild(style);
    }

    // Persistent settings
    if (!window.ytCaptureSettings) window.ytCaptureSettings = {
        promptText: 'Describe this video frame.',
        mode: 'prompt_run'  // 'prompt_run', 'prompt', 'capture'
    };
    const S = window.ytCaptureSettings;

    // Icon helper: circle with mode-dependent color/fill
    const captureIcon = (color, filled) => {
        const fill = filled ? color : 'none';
        return `<svg viewBox="0 0 16 16" width="16" height="16" style="vertical-align:middle"><circle cx="8" cy="8" r="5.5" stroke="${color}" stroke-width="1.5" fill="${fill}"/></svg>`;
    };
    const chevron = `<svg viewBox="0 0 24 24" width="12" height="12" fill="currentColor"><path fill-rule="evenodd" clip-rule="evenodd" d="M4.29289 8.29289C4.68342 7.90237 5.31658 7.90237 5.70711 8.29289L12 14.5858L18.2929 8.29289C18.6834 7.90237 19.3166 7.90237 19.7071 8.29289C20.0976 8.68342 20.0976 9.31658 19.7071 9.70711L12.7071 16.7071C12.3166 17.0976 11.6834 17.0976 11.2929 16.7071L4.29289 9.70711C3.90237 9.31658 3.90237 8.68342 4.29289 8.29289Z"/></svg>`;

    const modeStyles = {
        prompt_run: { color: '#ff5555', filled: true,  label: 'Send to Prompt &amp; Run' },
        prompt:     { color: '#ff5555', filled: false, label: 'Send to Prompt' },
        capture:    { color: '#22dd66', filled: true,  label: 'Send to Note' }
    };

    // Container
    const wrap = document.createElement('div');
    wrap.id = 'yt-capture-wrap';
    wrap.style.cssText = 'display:flex;position:relative;';

    // Main capture button
    const btn = document.createElement('button');
    btn.className = 'uk-btn uk-btn-icon uk-btn-sm text-lg uk-btn-default cursor-pointer';
    btn.style.cssText = 'border-radius:4px 0 0 4px;border-right:none;';
    btn.title = 'Capture video frame';

    // Drop arrow button
    const dd = document.createElement('button');
    dd.className = 'uk-btn uk-btn-icon uk-btn-sm text-lg uk-btn-default cursor-pointer';
    dd.style.cssText = 'border-radius:0 4px 4px 0;padding:0 4px;margin-left:0;';
    dd.innerHTML = chevron;
    dd.title = 'Capture settings';

    function updateMainIcon() {
        const m = modeStyles[S.mode];
        btn.innerHTML = captureIcon(m.color, m.filled);
    }
    updateMainIcon();

    // Dropdown panel
    const panel = document.createElement('div');
    panel.style.cssText = 'display:none;position:absolute;top:100%;right:0;background:white;border:1px solid #ccc;border-radius:4px;box-shadow:0 2px 8px rgba(0,0,0,0.15);z-index:10;min-width:220px;margin-top:2px;overflow:hidden;';

    function buildPanel() {
        let html = `<div style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">
            <textarea id="yt-prompt-text" rows="2" placeholder="Prompt text..."
                style="width:100%;background:#f9fafb;color:#333;border:1px solid #d1d5db;border-radius:4px;padding:6px 8px;font-size:12px;resize:vertical;font-family:inherit;box-sizing:border-box;">${S.promptText}</textarea>
        </div>`;
        for (const [mode, ms] of Object.entries(modeStyles)) {
            const active = mode === S.mode;
            const bg     = active ? '#f0f4ff' : '#fff';
            const weight = active ? '600' : '400';
            html += `<div class="yt-mode-item" data-mode="${mode}"
                style="padding:6px 12px;curs

# 2. YouTube Player Embed

Embeds a YouTube player via the IFrame API, defaulting to a specific video and playlist. The player instance is stored as `window.ytPlayer` so other code (e.g. the capture button) can query the current video ID and playback time. Set the video ID, and optionally the playlist ID, below.

In [ ]:
yt_video_id = 'aircAruvnKk'
yt_playlist_id = 'PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi'  # set to '' for no playlist
#yt_video_id = '8iGzBMboA0I'
#yt_playlist_id = 'PLtmWHNX-gukIc92m1K0P6bIOnZb-mg0hY'  # set to '' for no playlist

In [ ]:
from IPython.display import HTML

HTML(f"""
<div id="yt-player"></div>
<script>
var tag = document.createElement('script');
tag.src = "https://www.youtube.com/iframe_api";
document.head.appendChild(tag);

function createPlayer() {{
    window.ytPlayer?.destroy();
    window.ytPlayer = new YT.Player('yt-player', {{
        width: '720', height: '480',
        videoId: '{yt_video_id}',
        playerVars: {{{f"rel: 0, list: '{yt_playlist_id}'" if yt_playlist_id else 'rel: 0'}}}
    }});
}}
if (window.YT && YT.Player) createPlayer();
else window.onYouTubeIframeAPIReady = createPlayer;
</script>
""")

HTML(
<div id="yt-player"></div>
<script>
var tag = document.createElement('script');
tag.src = "https://www.youtube.com/iframe_api";
document.head.appendChild(tag);

function createPlayer() {
    window.ytPlayer?.destroy();
    window.ytPlayer = new YT.Player('yt-player', {
        width: '720', height: '480',
        videoId: 'aircAruvnKk',
        playerVars: {rel: 0, list: 'PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi'}
    });
}
if (window.YT && YT.Player) createPlayer();
else window.onYouTubeIframeAPIReady = createPlayer;
</script>
)

# 3. Notes